# Multimodal Fake News Detection Demo

In [2]:
import sys
sys.path.append("../src")

import numpy as np
import torch

from load_dataset import load_data, load_sample
from image_model import ImageModel

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

## Load Dataset

In [4]:
df = load_data()
df.head()

,text,image,score,comments,label
0,Aliens spotted flying over New York,placeholder.jpg,120,30,0
1,Government announces new economic policy,placeholder.jpg,560,210,1
2,Celebrity claims miracle cure for cancer,placeholder.jpg,300,45,0
3,Scientists discover new planet similar to Earth,placeholder.jpg,450,120,1


## Sample Data

In [3]:
text, image, metadata, label = load_sample(df, 0)

print("Text:", text)
print("Metadata:", metadata)
print("Label:", label)

image

tensor([[[0.9961, 1.0000, 0.9843,  ..., 0.9882, 0.9843, 0.9765],
         [0.9961, 1.0000, 0.9843,  ..., 0.9922, 0.9804, 0.9765],
         [1.0000, 1.0000, 0.9882,  ..., 0.9882, 0.9765, 0.9765],
         ...,
         [0.2980, 0.2980, 0.2980,  ..., 0.3059, 0.3137, 0.3098],
         [0.2980, 0.2980, 0.2980,  ..., 0.3059, 0.3137, 0.3137],
         [0.2980, 0.2980, 0.2980,  ..., 0.3059, 0.3137, 0.3137]],

        [[0.9725, 0.9882, 0.9922,  ..., 0.9922, 0.9882, 0.9922],
         [0.9686, 0.9882, 0.9882,  ..., 0.9922, 0.9882, 0.9922],
         [0.9725, 0.9882, 0.9961,  ..., 0.9961, 0.9922, 0.9922],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0078, 0.0196, 0.0196],
         [0.0000, 0.0000, 0.0000,  ..., 0.0078, 0.0157, 0.0157],
         [0.0000, 0.0000, 0.0000,  ..., 0.0078, 0.0157, 0.0157]],

        [[0.9255, 0.9569, 0.9882,  ..., 1.0000, 0.9961, 0.9961],
         [0.9333, 0.9686, 0.9922,  ..., 1.0000, 0.9961, 0.9961],
         [0.9569, 0.9843, 0.9961,  ..., 1.0000, 0.9961, 0.

## Image Feature Extraction

In [5]:
model = ImageModel()

image_features_list = []

for i in range(len(df)):
    _, image, _, _ = load_sample(df, i)

    image = image.unsqueeze(0)

    with torch.no_grad():
        features = model(image)

    image_features_list.append(features.squeeze().numpy())

X_image = np.array(image_features_list)

print("X_image shape:", X_image.shape)

X_image shape: (4, 2048)


### extract metadata + labels

In [6]:
metadata_list = []

for i in range(len(df)):
    _, _, metadata, _ = load_sample(df, i)
    metadata_list.append(metadata)

X_meta = np.array(metadata_list)
y = df["label"].values

print("X_meta shape:", X_meta.shape)
print("y shape:", y.shape)

X_meta shape: (4, 2)
y shape: (4,)


### combined features

In [8]:
X_combined = np.concatenate((X_image, X_meta), axis=1)

print("X_combined shape:", X_combined.shape)

X_combined shape: (4, 2050)


## Model Training

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.25, random_state=42
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 1.0


### Evaluation

In [11]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       1.00      1.00      1.00         1

    accuracy                           1.00         1
   macro avg       1.00      1.00      1.00         1
weighted avg       1.00      1.00      1.00         1



In [23]:
np.save("X_image.npy", X_image)
np.save("X_meta.npy", X_meta)
np.save("y.npy", y)